In [ ]:
# -*- coding: utf-8 -*-
# LightCycleGAN-LD: Lightweight GAN for Low-Dose CT Reconstruction
# Designed by Imam Ahasan

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image, make_grid
import numpy as np
import h5py
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure

# Optional W&B
try:
    import wandb
    WANDB_ENABLED = True
except ImportError:
    WANDB_ENABLED = False

# Paths & device
BASE_PATH = "/Users/imamahasan/MyData/Code/LightCycleGAN-LD"
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"Using device: {DEVICE}")

# Create necessary folders
for d in ["observation_train", "ground_truth_train", "checkpoints", "outputs", "logs"]:
    os.makedirs(os.path.join(BASE_PATH, d), exist_ok=True)

TRAIN_OBS_DIR  = os.path.join(BASE_PATH, "observation_train")
TRAIN_GT_DIR   = os.path.join(BASE_PATH, "ground_truth_train")
CHECKPOINT_DIR = os.path.join(BASE_PATH, "checkpoints")
OUTPUT_DIR     = os.path.join(BASE_PATH, "outputs")
LOG_DIR        = os.path.join(BASE_PATH, "logs")

# Hyperparameters
BATCH_SIZE        = 2
LR                = 1e-4
EPOCHS            = 100
LAMBDA_PERCEPTUAL = 1
LAMBDA_SSIM       = 5
LAMBDA_CYCLE      = 10
SAVE_SAMPLE_EVERY = 5

# ─── 1. DATASET ───────────────────────────────────────────────────────────────
class LoDoPaBDataset(Dataset):
    def __init__(self, obs_dir, gt_dir):
        self.obs_files = sorted(os.listdir(obs_dir))
        self.obs_dir   = obs_dir
        self.gt_dir    = gt_dir

    def __len__(self):
        return len(self.obs_files)

    def __getitem__(self, idx):
        obs_path = os.path.join(self.obs_dir, self.obs_files[idx])
        gt_path  = os.path.join(self.gt_dir,
                               self.obs_files[idx].replace("observation","ground_truth"))
        with h5py.File(obs_path,'r') as f1, h5py.File(gt_path,'r') as f2:
            obs = f1['data'][0].astype(np.float32)
            gt  = f2['data'][0].astype(np.float32)
        # normalize to [0,1]
        obs = (obs - obs.min())/(obs.max()-obs.min()+1e-8)
        gt  = (gt  -  gt.min())/(gt.max()-gt.min()+1e-8)
        return (
            torch.from_numpy(obs).unsqueeze(0),
            torch.from_numpy(gt).unsqueeze(0)
        )

# ─── 2. MODEL BLOCKS ─────────────────────────────────────────────────────────
class MultiKernelResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c,c,1)
        self.conv3 = nn.Conv2d(c,c,3,padding=1)
        self.conv5 = nn.Conv2d(c,c,5,padding=2)
        self.merge = nn.Conv2d(c*3,c,1)
        self.res   = nn.Conv2d(c,c,1)
    def forward(self,x):
        o = torch.cat([self.conv1(x),
                       self.conv3(x),
                       self.conv5(x)],1)
        return self.merge(o) + self.res(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1,64,3,padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(64,128,3,2,padding=1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(128,256,3,2,padding=1), nn.ReLU())
        self.bottleneck = MultiKernelResBlock(256)
        self.up1 = nn.ConvTranspose2d(256,128,2,2)
        self.up2 = nn.ConvTranspose2d(128,64,2,2)
        self.red3 = nn.Conv2d(256,128,1)
        self.red2 = nn.Conv2d(128,64,1)
        self.dec3 = MultiKernelResBlock(128)
        self.dec2 = MultiKernelResBlock(64)
        self.final = nn.Conv2d(64,1,1)
        self.tanh  = nn.Tanh()

    def forward(self,x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b  = self.bottleneck(e3)
        d3 = self.up1(b)
        s3 = F.interpolate(self.red3(e3), size=d3.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.dec3(d3 + s3)
        d2 = self.up2(d3)
        s2 = F.interpolate(self.red2(e2), size=d2.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(d2 + s2)
        return self.tanh(self.final(d2))

class Discriminator(nn.Module):
    def __init__(self, in_ch=1, base=64):
        super().__init__()
        def blk(i,o,s): return nn.Sequential(
            nn.Conv2d(i,o,4,s,1),
            nn.InstanceNorm2d(o),
            nn.LeakyReLU(0.2,inplace=True)
        )
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,base,4,2,1), nn.LeakyReLU(0.2,inplace=True),
            blk(base,base*2,2), blk(base*2,base*4,2),
            blk(base*4,base*8,1),
            nn.Conv2d(base*8,1,4,1,1)
        )
    def forward(self,x): return self.net(x)

class ResidualBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(c,c,3,padding=1), nn.InstanceNorm2d(c), nn.ReLU(inplace=True),
            nn.Conv2d(c,c,3,padding=1), nn.InstanceNorm2d(c)
        )
    def forward(self,x): return x + self.seq(x)

class StudentGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32, num_res=2):
        super().__init__()
        self.d1    = nn.Sequential(
            nn.Conv2d(in_ch,base,7,1,3),
            nn.InstanceNorm2d(base),
            nn.ReLU()
        )
        self.d2    = nn.Sequential(
            nn.Conv2d(base,base*2,4,2,1),
            nn.InstanceNorm2d(base*2),
            nn.ReLU()
        )
        self.res   = nn.Sequential(*[ResidualBlock(base*2) for _ in range(num_res)])
        self.u1    = nn.Sequential(
            nn.ConvTranspose2d(base*2,base,4,2,1),
            nn.InstanceNorm2d(base),
            nn.ReLU()
        )
        self.final = nn.Conv2d(base,out_ch,7,1,3)
        self.tanh  = nn.Tanh()

    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        r  = self.res(d2)
        u1 = self.u1(r)
        if u1.shape[2:] != d1.shape[2:]:
            u1 = F.interpolate(u1, size=d1.shape[2:], mode='bilinear', align_corners=False)
        return self.tanh(self.final(u1 + d1))

# ─── 3. LOSSES ────────────────────────────────────────────────────────────────
vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features[:16].to(DEVICE)
for p in vgg.parameters(): p.requires_grad=False

def perceptual_loss(fake, real):
    if fake.shape[2:] != real.shape[2:]:
        fake = F.interpolate(fake, size=real.shape[2:], mode='bilinear', align_corners=False)
    f3 = fake.repeat(1,3,1,1)
    r3 = real.repeat(1,3,1,1)
    return F.l1_loss(vgg(f3), vgg(r3))

ssim_m = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)
def ssim_loss(fake, real):
    if fake.shape[2:] != real.shape[2:]:
        fake = F.interpolate(fake, size=real.shape[2:], mode='bilinear', align_corners=False)
    return 1 - ssim_m(fake, real)

def cycle_loss(fake, real):
    rec = G(fake)
    if rec.shape[2:] != real.shape[2:]:
        rec = F.interpolate(rec, size=real.shape[2:], mode='bilinear', align_corners=False)
    return F.l1_loss(rec, real)

def distill_loss(teacher, student):
    if teacher.shape[2:] != student.shape[2:]:
        teacher = F.interpolate(teacher, size=student.shape[2:], mode='bilinear', align_corners=False)
    return F.l1_loss(student, teacher.detach())

# ─── 4. SETUP ─────────────────────────────────────────────────────────────────
G         = Generator().to(DEVICE)
D         = Discriminator().to(DEVICE)
student_G = StudentGenerator().to(DEVICE)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5,0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5,0.999))
opt_S = torch.optim.Adam(student_G.parameters(), lr=LR*0.5, betas=(0.5,0.999))

writer = SummaryWriter(LOG_DIR)
if WANDB_ENABLED:
    wandb.init(project="LightCycleGAN-LD",
               config=dict(lr=LR, batch_size=BATCH_SIZE, epochs=EPOCHS))

train_ds     = LoDoPaBDataset(TRAIN_OBS_DIR, TRAIN_GT_DIR)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)

# ─── 5. TRAINING LOOP ─────────────────────────────────────────────────────────
for epoch in range(EPOCHS):
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for i, (ldct, fdct) in enumerate(loop):
        ldct, fdct = ldct.to(DEVICE), fdct.to(DEVICE)

        # D-step
        fake = G(ldct)
        if fake.shape[2:] != fdct.shape[2:]:
            fake = F.interpolate(fake, size=fdct.shape[2:], mode='bilinear', align_corners=False)

        opt_D.zero_grad()
        real_out = D(fdct)
        fake_out = D(fake.detach())
        loss_D = 0.5 * (
            F.binary_cross_entropy_with_logits(real_out, torch.ones_like(real_out)) +
            F.binary_cross_entropy_with_logits(fake_out, torch.zeros_like(fake_out))
        )
        loss_D.backward()
        opt_D.step()

        # G-step
        opt_G.zero_grad()
        fake_out_for_G = D(fake)
        adv_loss = F.binary_cross_entropy_with_logits(fake_out_for_G, torch.ones_like(fake_out_for_G))
        pl    = perceptual_loss(fake, fdct)
        sl    = ssim_loss(fake, fdct)
        cl    = cycle_loss(fake, fdct)
        loss_G = adv_loss + LAMBDA_PERCEPTUAL*pl + LAMBDA_SSIM*sl + LAMBDA_CYCLE*cl
        loss_G.backward()
        opt_G.step()

        # Distillation-step
        opt_S.zero_grad()
        student_out = student_G(ldct)
        kd = distill_loss(fake, student_out)
        kd.backward()
        opt_S.step()

        # Logging
        step = epoch * len(train_loader) + i
        writer.add_scalar("Loss/D", loss_D.item(), step)
        writer.add_scalar("Loss/G", loss_G.item(), step)
        writer.add_scalar("Loss/KD", kd.item(), step)
        if WANDB_ENABLED:
            wandb.log({"loss_D": loss_D.item(),
                       "loss_G": loss_G.item(),
                       "loss_KD": kd.item()}, step=step)

        # ─ Sample images (FIXED) ─────────────────────────────────────
        if step % SAVE_SAMPLE_EVERY == 0:
            inp       = ldct[0:1]
            gen_img   = fake[0:1]
            target    = fdct[0:1]
            Hf, Wf    = gen_img.shape[2], gen_img.shape[3]
            inp_vis   = F.interpolate(inp, size=(Hf,Wf),
                                      mode='bilinear', align_corners=False)
            tgt_vis   = F.interpolate(target, size=(Hf,Wf),
                                      mode='bilinear', align_corners=False)
            grid      = make_grid(
                torch.cat([inp_vis, gen_img, tgt_vis], dim=0),
                nrow=3,
                normalize=True,
                scale_each=True
            )
            save_image(grid,
                       os.path.join(OUTPUT_DIR, f"epoch{epoch}_step{step}.png"))
            writer.add_image("Samples", grid, step)
            if WANDB_ENABLED:
                wandb.log({"Samples": wandb.Image(grid)}, step=step)

    # Checkpoints
    torch.save(G.state_dict(), os.path.join(CHECKPOINT_DIR, f"G_epoch{epoch}.pth"))
    torch.save(D.state_dict(), os.path.join(CHECKPOINT_DIR, f"D_epoch{epoch}.pth"))
    torch.save(student_G.state_dict(), os.path.join(CHECKPOINT_DIR, f"S_epoch{epoch}.pth"))

writer.close()
if WANDB_ENABLED:
    wandb.finish()

print("Training complete!")


✅ Using device: mps


Epoch 1/100:   0%|          | 0/140 [00:00<?, ?it/s]/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torch/autograd/__init__.py:251: UserWarning: The operator 'aten::sgn.out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:13.)
  Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch 47/100:  85%|████████▌ | 119/140 [01:39<00:17,  1.21it/s]